# Monster Spellcasting Analysis

This notebook analyzes which D&D 5e monsters have spellcasting abilities and what spells they can cast.

**Goal:** Understand monster spell usage to focus feature engineering efforts.

**Input:** `data/dnd5e_monsters_from_json.csv`

## Setup

In [25]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter
from pathlib import Path

# Detect execution context
cwd = Path.cwd()
if cwd.name == 'exploration':
    DATA_DIR = '../../data'
elif cwd.name == 'notebooks':
    DATA_DIR = '../data'
else:
    DATA_DIR = './data'

print(f"Data directory: {DATA_DIR}")

Data directory: ./data


## Load Monster Data

In [26]:
# Load monster data
df = pd.read_csv(f"{DATA_DIR}/dnd5e_monsters_from_json.csv")
print(f"Loaded {len(df)} monsters")
print(f"Columns: {list(df.columns)}")

Loaded 382 monsters
Columns: ['Name', 'Size', 'Type', 'Alignment', 'HP', 'AC', 'Speed', 'Challenge_Rating', 'XP', 'STR', 'STR_Mod', 'DEX', 'DEX_Mod', 'CON', 'CON_Mod', 'INT', 'INT_Mod', 'WIS', 'WIS_Mod', 'CHA', 'CHA_Mod', 'Saving_Throws', 'Skills', 'Vulnerabilities', 'Damage_Vulnerabilities', 'Resistances', 'Immunities', 'Condition_Immunities', 'Passive_Perception', 'Senses', 'Languages', 'Traits', 'Actions', 'Reactions', 'Legendary_Actions', 'Legendary_Actions_Num', 'Bonus_Actions', 'Source_Name', 'Image_URL']


## Parse Traits Column

The Traits column contains JSON with trait name and description.

In [27]:
def parse_traits(traits_str):
    """Parse traits JSON string into list of dicts."""
    if pd.isna(traits_str):
        return []
    try:
        return json.loads(traits_str)
    except (json.JSONDecodeError, TypeError):
        return []

def get_trait_by_name(traits, name):
    """Get a specific trait by name."""
    for trait in traits:
        if isinstance(trait, dict) and trait.get('Name') == name:
            return trait
    return None

# Parse traits for all monsters
df['traits_parsed'] = df['Traits'].apply(parse_traits)

# Check for spellcasting traits
df['has_spellcasting'] = df['traits_parsed'].apply(
    lambda traits: get_trait_by_name(traits, 'Spellcasting') is not None
)
df['has_innate_spellcasting'] = df['traits_parsed'].apply(
    lambda traits: get_trait_by_name(traits, 'Innate Spellcasting') is not None
)

print(f"Creatures with 'Spellcasting' trait: {df['has_spellcasting'].sum()}")
print(f"Creatures with 'Innate Spellcasting' trait: {df['has_innate_spellcasting'].sum()}")
print(f"Total with any spellcasting: {(df['has_spellcasting'] | df['has_innate_spellcasting']).sum()}")

Creatures with 'Spellcasting' trait: 12
Creatures with 'Innate Spellcasting' trait: 20
Total with any spellcasting: 32


In [28]:
# Extract spellcasting details from traits
def extract_spellcasting_details(row):
    """Extract spellcasting type, level, ability, save DC, and description."""
    traits = row['traits_parsed']
    
    result = {
        'spellcasting_type': None,
        'spellcaster_level': None,
        'spellcasting_ability': None,
        'spell_save_dc': None,
        'spell_desc': None
    }
    
    leveled_trait = get_trait_by_name(traits, 'Spellcasting')
    innate_trait = get_trait_by_name(traits, 'Innate Spellcasting')
    
    # Determine type
    if leveled_trait and innate_trait:
        result['spellcasting_type'] = 'both'
    elif leveled_trait:
        result['spellcasting_type'] = 'leveled'
    elif innate_trait:
        result['spellcasting_type'] = 'innate'
    else:
        return result
    
    # Extract from leveled spellcasting
    if leveled_trait:
        desc = leveled_trait.get('Desc', '')  # Field is 'Desc', not 'Description'
        result['spell_desc'] = desc
        
        # Extract spellcaster level: "The X is a Nth-level spellcaster"
        level_match = re.search(r'(\d+)(?:st|nd|rd|th)-level spellcaster', desc)
        if level_match:
            result['spellcaster_level'] = int(level_match.group(1))
        
        # Extract spellcasting ability
        ability_match = re.search(r'spellcasting ability is (\w+)', desc)
        if ability_match:
            result['spellcasting_ability'] = ability_match.group(1)
        
        # Extract save DC
        dc_match = re.search(r'spell save DC (\d+)', desc)
        if dc_match:
            result['spell_save_dc'] = int(dc_match.group(1))
    
    # Extract from innate spellcasting (may override or supplement)
    if innate_trait:
        desc = innate_trait.get('Desc', '')  # Field is 'Desc', not 'Description'
        if not result['spell_desc']:
            result['spell_desc'] = desc
        else:
            result['spell_desc'] = result['spell_desc'] + ' ' + desc
        
        # Extract spellcasting ability if not already set
        if not result['spellcasting_ability']:
            ability_match = re.search(r'spellcasting ability is (\w+)', desc)
            if ability_match:
                result['spellcasting_ability'] = ability_match.group(1)
        
        # Extract save DC if not already set
        if not result['spell_save_dc']:
            dc_match = re.search(r'spell save DC (\d+)', desc)
            if dc_match:
                result['spell_save_dc'] = int(dc_match.group(1))
    
    return result

# Apply to all creatures
spellcasting_details = df.apply(extract_spellcasting_details, axis=1, result_type='expand')
df = pd.concat([df, spellcasting_details], axis=1)

print(f"Spellcasting types:")
print(df['spellcasting_type'].value_counts(dropna=False).head(10))

Spellcasting types:
spellcasting_type
None       350
innate      20
leveled     12
Name: count, dtype: int64


## Identify Spellcasters

In [29]:
# Create spellcasters dataframe
df_spellcasters = df[df['spellcasting_type'].notna()][[
    'Name', 'Challenge_Rating', 'Type', 
    'spellcasting_type', 'spellcaster_level', 'spellcasting_ability', 'spell_save_dc',
    'spell_desc'
]].copy()

print(f"Total spellcasters: {len(df_spellcasters)}")
print(f"\nBy type:")
print(df_spellcasters['spellcasting_type'].value_counts())

Total spellcasters: 32

By type:
spellcasting_type
innate     20
leveled    12
Name: count, dtype: int64


In [30]:
# Show leveled spellcasters
print("=== Leveled Spellcasters ===")
leveled = df_spellcasters[df_spellcasters['spellcasting_type'].isin(['leveled', 'both'])]
print(leveled[['Name', 'Challenge_Rating', 'spellcaster_level', 'spellcasting_ability']].to_string(index=False))

=== Leveled Spellcasters ===
         Name Challenge_Rating  spellcaster_level spellcasting_ability
      Acolyte              1/4                1.0               Wisdom
  Androsphinx               17               12.0               Wisdom
     Archmage               12               18.0         Intelligence
 Cult Fanatic                2                4.0                 None
        Druid                2                4.0               Wisdom
Guardian Naga               10               11.0               Wisdom
   Gynosphinx               11                9.0         Intelligence
         Lich               21               18.0         Intelligence
         Mage                6                9.0         Intelligence
   Mummy Lord               15               10.0               Wisdom
       Priest                2                5.0               Wisdom
  Spirit Naga                8               10.0         Intelligence


In [31]:
# Show innate spellcasters
print("=== Innate Spellcasters ===")
innate = df_spellcasters[df_spellcasters['spellcasting_type'].isin(['innate', 'both'])]
print(innate[['Name', 'Challenge_Rating', 'spellcasting_ability', 'spell_save_dc']].to_string(index=False))

=== Innate Spellcasters ===
                    Name Challenge_Rating spellcasting_ability  spell_save_dc
             Cloud Giant                9             Charisma            NaN
                  Couatl                4             Charisma           14.0
Deep Gnome (Svirfneblin)              1/2         Intelligence           11.0
                    Deva               10             Charisma           17.0
                  Djinni               11             Charisma           17.0
                  Drider                6               Wisdom           13.0
                    Drow              1/4             Charisma           11.0
                   Dryad                1             Charisma           14.0
                 Efreeti               11                 None           15.0
                Glabrezu                9         Intelligence           16.0
               Green Hag                3             Charisma           12.0
                   Lamia            

## Extract Spell Lists

In [32]:
def normalize_spell_name(spell):
    """Normalize spell name to match spell database."""
    spell = spell.strip().lower()
    # Handle 'power word stun' -> 'power word: stun'
    spell = re.sub(r'^power word (\w+)$', r'power word: \1', spell)
    # Handle 'blindness deafness' -> 'blindness/deafness'
    spell = re.sub(r'^blindness deafness$', 'blindness/deafness', spell)
    return spell

def extract_spells_from_desc(desc):
    """Extract spell names from spellcasting description."""
    if not desc:
        return []
    
    spells = []
    
    # For leveled spellcasting: split by bullets
    if '•' in desc:
        # Find the first bullet and start from there
        first_bullet = desc.find('•')
        spell_section = desc[first_bullet:]
        
        categories = re.split(r'[•]', spell_section)
        for category in categories:
            # Remove level markers like "Cantrips (at will):" or "1st level (4 slots):"
            category = re.sub(r'^\s*(?:Cantrips?|\d+(?:st|nd|rd|th)\s+level)\s*\([^)]+\):\s*', '', category, flags=re.IGNORECASE)
            for part in category.split(','):
                spell = normalize_spell_name(part)
                spell = re.sub(r'\*.*$', '', spell).strip()
                spell = re.sub(r'\s*\([^)]*\)', '', spell).strip()
                if len(spell) >= 3:
                    spells.append(spell)
    else:
        # For innate spellcasting - split by frequency markers
        freq_pattern = r'(?:At will|\d+/day(?:\s+each)?):\s*'
        parts = re.split(freq_pattern, desc, flags=re.IGNORECASE)
        
        for part in parts:
            # Skip preamble (first part before any frequency marker)
            if 'spellcasting ability' in part.lower():
                continue
            part = re.sub(r'\s+\d+/day.*$', '', part, flags=re.IGNORECASE)
            for spell_str in re.split(r',\s*', part):
                spell = normalize_spell_name(spell_str)
                spell = re.sub(r'\s*\([^)]*\)', '', spell).strip()
                if len(spell) >= 3:
                    spells.append(spell)
    
    # Filter out non-spells
    skip_patterns = [
        'the following', 'requiring', 'material', 'components', 'verbal',
        'level', 'slot', 'combat', 'cast', 'spellcaster', ' is ', ' can ',
        'each:', 'elemental', 'prepared', 'to hit'
    ]
    
    cleaned = []
    for s in spells:
        s = s.strip()
        if not s or len(s) < 3:
            continue
        if any(p in s for p in skip_patterns):
            continue
        if re.match(r'^\d', s) or re.match(r'^[•\-]', s):
            continue
        if s not in cleaned:
            cleaned.append(s)
    
    return cleaned

# Extract spells for each spellcaster
df_spellcasters['spell_list'] = df_spellcasters['spell_desc'].apply(extract_spells_from_desc)
df_spellcasters['spell_count'] = df_spellcasters['spell_list'].apply(len)

print(f"Total spells extracted: {df_spellcasters['spell_list'].apply(len).sum()}")
print(f"\nSpell counts per creature:")
print(df_spellcasters[['Name', 'spell_count']].sort_values('spell_count', ascending=False).head(10).to_string(index=False))

Total spells extracted: 304

Spell counts per creature:
         Name  spell_count
         Lich           26
     Archmage           25
         Mage           16
  Androsphinx           15
Guardian Naga           15
   Gynosphinx           15
   Mummy Lord           15
       Couatl           13
     Rakshasa           13
  Spirit Naga           13


In [33]:
# Show spell lists for a few example creatures
examples = ['Archmage', 'Couatl', 'Acolyte', 'Drow']
for name in examples:
    creature = df_spellcasters[df_spellcasters['Name'] == name]
    if len(creature) > 0:
        row = creature.iloc[0]
        print(f"\n=== {name} ({row['spellcasting_type']}) ===")
        print(f"Spells: {row['spell_list']}")


=== Archmage (leveled) ===
Spells: ['fire bolt', 'light', 'mage hand', 'prestidigitation', 'shocking grasp', 'detect magic', 'identify', 'mage armor', 'magic missile', 'detect thoughts', 'mirror image', 'misty step', 'counterspell', 'fly', 'lightning bolt', 'banishment', 'fire shield', 'stoneskin', 'cone of cold', 'scrying', 'wall of force', 'globe of invulnerability', 'teleport', 'mind blank', 'time stop']

=== Couatl (innate) ===
Spells: ['detect evil and good', 'detect magic', 'detect thoughts', 'bless', 'create food and water', 'cure wounds', 'lesser restoration', 'protection from poison', 'sanctuary', 'shield', 'dream', 'greater restoration', 'scrying']

=== Acolyte (leveled) ===
Spells: ['light', 'sacred flame', 'thaumaturgy', 'bless', 'cure wounds', 'sanctuary']

=== Drow (innate) ===
Spells: ['dancing lights', 'darkness', 'faerie fire']


## Spell Frequency Analysis

In [34]:
# Count spell frequency across all creatures
all_spells = []
spell_to_creatures = {}

for _, row in df_spellcasters.iterrows():
    for spell in row['spell_list']:
        all_spells.append(spell)
        if spell not in spell_to_creatures:
            spell_to_creatures[spell] = []
        spell_to_creatures[spell].append(row['Name'])

# Create spell frequency dataframe
spell_counts = Counter(all_spells)
df_spell_counts = pd.DataFrame([
    {
        'spell_name': spell,
        'creature_count': count,
        'creatures': ', '.join(spell_to_creatures[spell])
    }
    for spell, count in spell_counts.items()
]).sort_values('creature_count', ascending=False)

print(f"Unique spells found: {len(df_spell_counts)}")
print(f"\nTop 20 most common spells among monsters:")
print(df_spell_counts.head(20).to_string(index=False))

Unique spells found: 127

Top 20 most common spells among monsters:
          spell_name  creature_count                                                                                                                                               creatures
        detect magic              15 Androsphinx, Archmage, Cloud Giant, Couatl, Djinni, Efreeti, Glabrezu, Gynosphinx, Lich, Mage, Night Hag, Pit Fiend, Rakshasa, Spirit Naga, Storm Giant
               light               7                                                                                 Acolyte, Archmage, Cloud Giant, Cult Fanatic, Mage, Priest, Storm Giant
        invisibility               7                                                                                                   Djinni, Efreeti, Lich, Oni, Planetar, Rakshasa, Solar
detect evil and good               6                                                                                                     Androsphinx, Couatl, Deva, Planetar, So

In [35]:
# Show spells used by only one creature (unique spells)
unique_spells = df_spell_counts[df_spell_counts['creature_count'] == 1]
print(f"\nSpells unique to one creature: {len(unique_spells)}")
print(unique_spells[['spell_name', 'creatures']].head(20).to_string(index=False))


Spells unique to one creature: 53
                   spell_name                creatures
                    wind walk                   Djinni
               inflict wounds             Cult Fanatic
at will: detect evil and good                   Djinni
                         blur Deep Gnome (Svirfneblin)
           blindness/deafness Deep Gnome (Svirfneblin)
                    stoneskin                 Archmage
              spare the dying              Androsphinx
               shocking grasp                 Archmage
                heroes' feast              Androsphinx
                zone of truth              Androsphinx
                        dream                   Couatl
       protection from poison                   Couatl
                 nondetection Deep Gnome (Svirfneblin)
                     teleport                 Archmage
                wall of force                 Archmage
                  fire shield                 Archmage
                   mind blank 

## Cross-Reference with Spell Data

In [36]:
# Load spell data
df_spells = pd.read_csv(f"{DATA_DIR}/spells.csv")
print(f"Loaded {len(df_spells)} spells from spell database")

# Normalize spell names for matching
df_spells['spell_name_lower'] = df_spells['spell_name'].str.lower().str.strip()

# Join spell counts with spell details
df_spell_details = df_spell_counts.merge(
    df_spells[['spell_name_lower', 'level', 'school', 'avg_damage', 'damage_type', 'is_aoe', 'estimated_targets']],
    left_on='spell_name',
    right_on='spell_name_lower',
    how='left'
)

# Check match rate
matched = df_spell_details['level'].notna().sum()
print(f"\nMatched {matched}/{len(df_spell_details)} spells ({matched/len(df_spell_details)*100:.1f}%)")

Loaded 574 spells from spell database

Matched 125/127 spells (98.4%)


In [37]:
# Show unmatched spells (might need name normalization)
unmatched = df_spell_details[df_spell_details['level'].isna()]['spell_name'].tolist()
if unmatched:
    print("Unmatched spells (may need name normalization):")
    for spell in unmatched[:20]:
        print(f"  - {spell}")

Unmatched spells (may need name normalization):
  - at will: detect evil and good
  - acid arrow


In [38]:
# Show damage spells that monsters can cast
damage_spells = df_spell_details[(df_spell_details['avg_damage'] > 0) & df_spell_details['level'].notna()]
print(f"\n=== Damage Spells Used by Monsters ({len(damage_spells)}) ===")
print(damage_spells[['spell_name', 'creature_count', 'level', 'avg_damage', 'damage_type', 'is_aoe']].sort_values('creature_count', ascending=False).to_string(index=False))


=== Damage Spells Used by Monsters (33) ===
      spell_name  creature_count  level  avg_damage damage_type is_aoe
    sacred flame               6    0.0         4.5     radiant  False
   magic missile               4    1.0         3.5       force  False
    cone of cold               3    5.0        36.0        cold   True
    flame strike               3    5.0        14.0        fire   True
spiritual weapon               3    2.0         4.5 unspecified  False
           sleep               3    1.0        22.5 unspecified  False
        fireball               3    3.0        28.0        fire   True
     thunderwave               3    1.0         9.0     thunder   True
            geas               2    5.0        27.5     psychic  False
       fire bolt               2    0.0         5.5        fire  False
  lightning bolt               2    3.0        28.0   lightning   True
    guiding bolt               2    1.0        14.0     radiant  False
  dimension door               2

In [39]:
# Breakdown by spell school
print("\n=== Monster Spells by School ===")
school_counts = df_spell_details.groupby('school')['creature_count'].sum().sort_values(ascending=False)
print(school_counts)


=== Monster Spells by School ===
school
Evocation        62
Divination       46
Abjuration       44
Transmutation    43
Enchantment      36
Conjuration      29
Illusion         27
Necromancy       15
Name: creature_count, dtype: int64


In [40]:
# Breakdown by spell level
print("\n=== Monster Spells by Level ===")
level_counts = df_spell_details.groupby('level')['creature_count'].sum().sort_index()
print(level_counts)


=== Monster Spells by Level ===
level
0.0    50
1.0    76
2.0    53
3.0    41
4.0    20
5.0    34
6.0    10
7.0     8
8.0     8
9.0     2
Name: creature_count, dtype: int64


## Summary DataFrames

Key dataframes for further analysis:

In [41]:
print("=== df_spellcasters ===")
print(f"Creatures with spellcasting abilities")
print(f"Shape: {df_spellcasters.shape}")
print(f"Columns: {list(df_spellcasters.columns)}")
print()

print("=== df_spell_counts ===")
print(f"Spell frequency among monsters")
print(f"Shape: {df_spell_counts.shape}")
print(f"Columns: {list(df_spell_counts.columns)}")
print()

print("=== df_spell_details ===")
print(f"Spell frequency with spell metadata")
print(f"Shape: {df_spell_details.shape}")
print(f"Columns: {list(df_spell_details.columns)}")

=== df_spellcasters ===
Creatures with spellcasting abilities
Shape: (32, 10)
Columns: ['Name', 'Challenge_Rating', 'Type', 'spellcasting_type', 'spellcaster_level', 'spellcasting_ability', 'spell_save_dc', 'spell_desc', 'spell_list', 'spell_count']

=== df_spell_counts ===
Spell frequency among monsters
Shape: (127, 3)
Columns: ['spell_name', 'creature_count', 'creatures']

=== df_spell_details ===
Spell frequency with spell metadata
Shape: (127, 10)
Columns: ['spell_name', 'creature_count', 'creatures', 'spell_name_lower', 'level', 'school', 'avg_damage', 'damage_type', 'is_aoe', 'estimated_targets']


In [42]:
# Display full spellcasters dataframe
df_spellcasters[['Name', 'Challenge_Rating', 'Type', 'spellcasting_type', 'spellcaster_level', 'spell_count']].sort_values('Challenge_Rating')

,Name,Challenge_Rating,Type,spellcasting_type,spellcaster_level,spell_count
109,Dryad,1,fey,innate,NaN,6
95,Deep Gnome (Svirfneblin),1/2,humanoid (gnome),innate,NaN,4
1,Acolyte,1/4,humanoid (any race),leveled,1.0,6
107,Drow,1/4,humanoid (elf),innate,NaN,3
97,Deva,10,celestial,innate,NaN,3
183,Guardian Naga,10,monstrosity,leveled,11.0,15
184,Gynosphinx,11,monstrosity,leveled,9.0,15
100,Djinni,11,elemental,innate,NaN,10
114,Efreeti,11,elemental,innate,NaN,8
36,Archmage,12,humanoid (any race),leveled,18.0,25


In [43]:
# Display spell frequency dataframe
df_spell_counts[df_spell_counts['spell_name']=='bane']

,spell_name,creature_count,creatures


In [44]:
df_spell_details[df_spell_details['spell_name']=='fireball'].iloc[0]

spell_name                        fireball
creature_count                           3
creatures            Lich, Mage, Pit Fiend
spell_name_lower                  fireball
level                                  3.0
school                           Evocation
avg_damage                            28.0
damage_type                           fire
is_aoe                                True
estimated_targets                      7.5
Name: 36, dtype: object

In [45]:
df_spellcasters.head()

,Name,Challenge_Rating,Type,spellcasting_type,spellcaster_level,spellcasting_ability,spell_save_dc,spell_desc,spell_list,spell_count
1,Acolyte,1/4,humanoid (any race),leveled,1.0,Wisdom,12.0,The acolyte is a 1st-level spellcaster. Its sp...,"[light, sacred flame, thaumaturgy, bless, cure...",6
24,Androsphinx,17,monstrosity,leveled,12.0,Wisdom,18.0,The sphinx is a 12th-level spellcaster. Its sp...,"[sacred flame, spare the dying, thaumaturgy, c...",15
36,Archmage,12,humanoid (any race),leveled,18.0,Intelligence,17.0,The archmage is an 18th-level spellcaster. Its...,"[fire bolt, light, mage hand, prestidigitation...",25
82,Cloud Giant,9,giant,innate,NaN,Charisma,NaN,The giant's innate spellcasting ability is Cha...,"[detect magic, fog cloud, light, feather fall,...",9
87,Couatl,4,celestial,innate,NaN,Charisma,14.0,The couatl's spellcasting ability is Charisma ...,"[detect evil and good, detect magic, detect th...",13


In [46]:
print(df_spells[df_spells['spell_name_lower']=='fireball']['description'].iloc[0])

A bright streak flashes from your pointing finger to a point you choose within range then blossoms with a low roar into an explosion of flame. Each creature in a 20-foot radius must make a Dexterity saving throw. A target takes 8d6 fire damage on a failed save, or half as much damage on a successful one. The fire spreads around corners. It ignites flammable objects in the area that aren’t being worn or carried.


# Exports

In [55]:

expanded_spellcasters = df_spellcasters.explode('spell_list').rename(columns={'spell_list': 'spell'}).dropna(subset=['spell'])
expanded_spellcasters = expanded_spellcasters[['Name', 'spell']].reset_index(drop=True)

save_path = f"{DATA_DIR}/spellcasters_spells.csv"
expanded_spellcasters.to_csv(save_path, index=False)

In [49]:
expanded_spellcasters.head()

,Name,spell
0,Acolyte,light
1,Acolyte,sacred flame
2,Acolyte,thaumaturgy
3,Acolyte,bless
4,Acolyte,cure wounds


In [54]:
# make sure to re-run spell_feature_engineering, then re-load df_spells
df_spells_joinable = df_spells.copy()
spell_columns = ['spell_name_lower', 'level', 
        'avg_damage', 'target_count', 'is_aoe',
       'modified_dpr', 'inflicts_blinded',
       'inflicts_charmed', 'inflicts_deafened', 'inflicts_frightened',
       'inflicts_incapacitated', 'inflicts_paralyzed', 'inflicts_petrified',
       'inflicts_poisoned', 'inflicts_prone', 'inflicts_restrained',
       'inflicts_stunned', 'grants_flying', 'ac_bonus', 'attack_bonus',
       'save_bonus', 'grants_advantage', 'inflicts_disadvantage',
       ]
df_spells_joinable = df_spells_joinable[spell_columns]
df_spells_joinable

,spell_name_lower,level,avg_damage,target_count,is_aoe,modified_dpr,inflicts_blinded,inflicts_charmed,inflicts_deafened,inflicts_frightened,...,inflicts_poisoned,inflicts_prone,inflicts_restrained,inflicts_stunned,grants_flying,ac_bonus,attack_bonus,save_bonus,grants_advantage,inflicts_disadvantage
0,acid splash,0,3.5,1.0,False,3.5,False,False,False,False,...,False,False,False,False,False,0,0.0,0.0,False,False
1,blade ward,0,0.0,1.0,False,0.0,False,False,False,False,...,False,False,False,False,False,0,0.0,0.0,False,False
2,booming blade,0,4.5,1.0,False,4.5,False,False,False,False,...,False,False,False,False,False,0,0.0,0.0,False,False
3,chill touch,0,4.5,1.0,False,4.5,False,False,False,False,...,False,False,False,False,False,0,0.0,0.0,True,True
4,control flames,0,0.0,NaN,True,0.0,False,False,False,False,...,False,False,False,False,False,0,0.0,0.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
569,time stop,9,0.0,1.0,False,0.0,False,False,False,False,...,False,False,False,False,False,0,0.0,0.0,False,False
570,true polymorph,9,0.0,1.0,False,0.0,False,False,False,False,...,False,False,False,False,False,0,0.0,0.0,False,False
571,true resurrection,9,0.0,1.0,False,0.0,False,False,False,False,...,False,False,False,False,False,0,0.0,0.0,False,False
572,weird,9,22.0,NaN,True,374.0,False,False,False,False,...,False,False,False,False,False,0,0.0,0.0,False,False
